In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, itertools, pathlib
warnings.filterwarnings('ignore')
pathlib.Path('figs').mkdir(exist_ok=True)
_TAG = 'nb4'
_fig_counter = itertools.count(1)
def _save_show(*a, **k):
    import matplotlib.pyplot as _plt
    for _n in _plt.get_fignums():
        _plt.figure(_n).savefig('figs/%s_fig%02d.png' % (_TAG, next(_fig_counter)), dpi=140, bbox_inches='tight')
    _plt.close('all')
plt.show = _save_show


# Characterizing Orbital Perturbations via Physics-Informed Neural Networks  
## (Mentor-provided) Implementing Real-Data Atmospheric Drag PINN with Mass, Reference Area, Density, and Ballistic Coefficient

**Project title:** *Characterizing Orbital Perturbations via Physics-Informed Neural Networks: An Inverse Problem Approach to Trajectory Reconstruction and Physical Parameter Discovery from Sparse Observation Data*

This notebook is the next technical step after the synthetic-orbit and first-PINN notebooks. The earlier notebooks treated drag as either a simplified perturbation term or an effective learned coefficient. This notebook upgrades the model to a more physically meaningful atmospheric drag formulation:

\[
\mathbf{a}_{drag} = -\frac{1}{2}\rho(h)\left(\frac{C_D A}{m}\right)\lVert \mathbf{v}_{rel}\rVert \mathbf{v}_{rel}
\]

where:

- \(\rho(h)\) is atmospheric density as a function of altitude,
- \(C_D\) is the drag coefficient,
- \(A\) is reference area,
- \(m\) is satellite mass,
- \(C_D A / m\) is the ballistic-coefficient-like drag parameter used in this notebook,
- \(\mathbf{v}_{rel}\) is the satellite velocity relative to the rotating atmosphere.

The goal is not just to predict the trajectory. The goal is to test whether a PINN can reconstruct a sparse real-data-derived orbit while learning or refining a physically interpretable drag parameter.

## 1. What this notebook does

This notebook is designed to be student-friendly but technically serious. It does five things:

1. Loads or downloads real satellite orbital data from CelesTrak.
2. Converts the data into Cartesian position and velocity states using SGP4.
3. Builds a sparse observation dataset from those real-data-derived states.
4. Trains a PINN whose physics loss includes gravity, the \(J_2\) oblateness perturbation, atmospheric rotation, atmospheric density, satellite mass, reference area, and drag coefficient.
5. Saves diagnostic plots, reconstruction tables, training history, and learned parameter summaries for the final paper.

### Important scientific caveat

TLE and SGP4 data are not raw GPS measurements. A TLE is already an estimated orbital element set produced from tracking data, and SGP4 is the standard model used to propagate it. In this notebook, we use CelesTrak/SGP4 output as a **real-data-derived benchmark trajectory**. That is acceptable for a high school research project if the final paper clearly says that the real-data stage uses TLE-derived state vectors rather than raw tracking measurements.

### Why this still matters

The scientific value is that the PINN is no longer trained only on synthetic equations. It now has to reconcile sparse observations derived from a real satellite data product with a physical model that includes atmospheric drag.

## 2. Install and import packages

This notebook is intended to run in Google Colab or a normal Jupyter environment. If the package installation cell fails in a local environment, install the packages manually in the terminal.

In [2]:
# [neutralized for headless run]
# # If you are running in Google Colab, this cell installs the only package
# # that is usually missing by default.
# #
# # sgp4: converts TLEs into position and velocity states.
# 
# %pip -q install sgp4

In [3]:
import math
import time
import json
import warnings
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

try:
    from scipy.integrate import solve_ivp
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except Exception as e:
    TORCH_AVAILABLE = False
    print("PyTorch is not available:", e)

from sgp4.api import Satrec, jday

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", DEVICE)
else:
    DEVICE = None

Using device: cpu


## 3. Set project paths

The earlier notebooks saved CSV files such as:

- `notebook2_celestrak_iss_standardized.csv`
- `synthetic_dense_truth.csv`
- `synthetic_sparse_observations.csv`

This notebook first tries to find the real standardized CelesTrak file from Notebook 2. If it does not exist, it downloads a fresh TLE from CelesTrak and propagates it into state vectors.

If working in a shared Google Drive folder, set `PROJECT_DIR` to that folder. Otherwise, the notebook will use the folder where it is currently running.

In [4]:
# Option A: use the current working directory.
PROJECT_DIR = Path.cwd()

# Option B: in Google Colab, uncomment and edit these lines.
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/YOUR_SHARED_PROJECT_FOLDER')

DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "pinn_outputs_real_drag"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR.resolve())
print("OUTPUT_DIR :", OUTPUT_DIR.resolve())

PROJECT_DIR: /private/tmp/pinn_run/nb4
OUTPUT_DIR : /private/tmp/pinn_run/nb4/pinn_outputs_real_drag


## 4. Define the real-data target satellite

For the first real-data drag PINN, a low-Earth-orbit object is best because atmospheric drag is most relevant in LEO. The International Space Station is a convenient first target because it is easy to find in public TLE catalogs and has a low enough altitude for atmospheric drag to matter.

The satellite physical parameters below are treated as **model assumptions**. They should be stated clearly in the final paper. A future version can replace these approximate values with a better satellite-specific mass and cross-sectional area from spacecraft documentation.

The most important combined parameter is:

\[
B = \frac{C_D A}{m}
\]

This notebook learns \(B\), not mass and area independently. This is because trajectory decay only strongly constrains the product \(C_D A / m\), not each component separately.

In [5]:
# CelesTrak/NORAD catalog ID for ISS (ZARYA)
TARGET_CATNR = 25544
TARGET_NAME = "ISS (ZARYA)"

# Approximate physical assumptions.
# These are not constants of nature. They are modeling assumptions.
SATELLITE_MASS_KG = 420000.0      # approximate mass scale for ISS; update if using another satellite
REFERENCE_AREA_M2 = 1000.0        # effective cross-sectional area depends on attitude; this is an assumption
DRAG_COEFFICIENT = 2.2            # common first-order assumption for spacecraft drag

PRIOR_BALLISTIC_COEFF = DRAG_COEFFICIENT * REFERENCE_AREA_M2 / SATELLITE_MASS_KG

print("Target satellite:", TARGET_NAME)
print("Mass assumption [kg]:", SATELLITE_MASS_KG)
print("Reference area assumption [m^2]:", REFERENCE_AREA_M2)
print("Drag coefficient assumption:", DRAG_COEFFICIENT)
print("Prior B = Cd*A/m [m^2/kg]:", PRIOR_BALLISTIC_COEFF)

Target satellite: ISS (ZARYA)
Mass assumption [kg]: 420000.0
Reference area assumption [m^2]: 1000.0
Drag coefficient assumption: 2.2
Prior B = Cd*A/m [m^2/kg]: 0.005238095238095238


## 5. Download or load real-data-derived state vectors

This notebook uses two possible data paths:

### Preferred path

Use the standardized real-data file made by Notebook 2:

`notebook2_celestrak_iss_standardized.csv`

### Backup path

Download a current TLE from CelesTrak and propagate it with SGP4 into Cartesian state vectors.

### Offline fallback

If neither path is available, the notebook uses a hard-coded historical ISS TLE only so that the code structure can be tested. This fallback should **not** be used for final results unless the paper explicitly says it used the fallback TLE.

In [6]:
def timestamp_to_jday(ts):
    """Convert a pandas UTC timestamp to Julian day + fractional day for sgp4."""
    ts = pd.Timestamp(ts).tz_convert("UTC")
    sec = ts.second + ts.microsecond / 1e6 + ts.nanosecond / 1e9
    return jday(ts.year, ts.month, ts.day, ts.hour, ts.minute, sec)


def jd_to_timestamp_utc(jd):
    """Convert Julian day to pandas UTC timestamp."""
    unix_seconds = (jd - 2440587.5) * 86400.0
    return pd.to_datetime(unix_seconds, unit="s", utc=True)


def parse_tle_block(tle_text):
    """Parse a 2-line or 3-line TLE block."""
    lines = [line.strip() for line in tle_text.splitlines() if line.strip()]
    if len(lines) >= 3 and lines[1].startswith("1 ") and lines[2].startswith("2 "):
        return lines[0], lines[1], lines[2]
    if len(lines) >= 2 and lines[0].startswith("1 ") and lines[1].startswith("2 "):
        return "UNKNOWN_OBJECT", lines[0], lines[1]
    raise ValueError("Could not parse TLE block.")


def propagate_tle_to_states(tle_name, tle_line1, tle_line2, minutes_step=5, total_hours=12):
    """Propagate a TLE into a dense table of Cartesian states."""
    sat = Satrec.twoline2rv(tle_line1, tle_line2)
    tle_epoch_jd = sat.jdsatepoch + sat.jdsatepochF
    tle_epoch_ts = jd_to_timestamp_utc(tle_epoch_jd)

    propagation_minutes = np.arange(0, total_hours * 60 + minutes_step, minutes_step)
    propagation_times = tle_epoch_ts + pd.to_timedelta(propagation_minutes, unit="m")

    records = []
    for ts in propagation_times:
        jd, fr = timestamp_to_jday(ts)
        error_code, r_km, v_kms = sat.sgp4(jd, fr)
        records.append({
            "timestamp_utc": pd.Timestamp(ts).tz_convert("UTC"),
            "error_code": error_code,
            "x_km": r_km[0] if error_code == 0 else np.nan,
            "y_km": r_km[1] if error_code == 0 else np.nan,
            "z_km": r_km[2] if error_code == 0 else np.nan,
            "vx_kms": v_kms[0] if error_code == 0 else np.nan,
            "vy_kms": v_kms[1] if error_code == 0 else np.nan,
            "vz_kms": v_kms[2] if error_code == 0 else np.nan,
        })

    df = pd.DataFrame(records)
    df = df[df["error_code"] == 0].copy()
    df["dataset_name"] = "celestrak_tle_sgp4_dense"
    df["object_name"] = tle_name
    df["source_name"] = "CelesTrak TLE propagated with SGP4"
    df["t_sec"] = (df["timestamp_utc"] - df["timestamp_utc"].iloc[0]).dt.total_seconds()
    df["radius_km"] = np.linalg.norm(df[["x_km", "y_km", "z_km"]].to_numpy(), axis=1)
    df["speed_kms"] = np.linalg.norm(df[["vx_kms", "vy_kms", "vz_kms"]].to_numpy(), axis=1)
    df["altitude_km"] = df["radius_km"] - 6378.1363
    return df.reset_index(drop=True), sat


def download_celestrak_tle(catnr):
    """Download a TLE from CelesTrak for one catalog number."""
    url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={catnr}&FORMAT=TLE"
    print("Downloading:", url)
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.text

In [7]:
# 1. Try to load Notebook 2 output.
notebook2_candidates = [
    DATA_DIR / "notebook2_celestrak_iss_standardized.csv",
    PROJECT_DIR / "data" / "notebook2_celestrak_iss_standardized.csv",
]

real_dense_df = None
real_data_source_note = None
satrec_object = None

for candidate in notebook2_candidates:
    if candidate.exists():
        print("Loading existing Notebook 2 standardized real-data file:", candidate)
        real_dense_df = pd.read_csv(candidate)
        if "timestamp_utc" in real_dense_df.columns:
            real_dense_df["timestamp_utc"] = pd.to_datetime(real_dense_df["timestamp_utc"], utc=True)
        real_data_source_note = f"Loaded from {candidate.name}"
        break

# 2. If not available, download a fresh TLE from CelesTrak.
if real_dense_df is None:
    try:
        tle_text = download_celestrak_tle(TARGET_CATNR)
        tle_name, tle_l1, tle_l2 = parse_tle_block(tle_text)
        real_dense_df, satrec_object = propagate_tle_to_states(
            tle_name=tle_name,
            tle_line1=tle_l1,
            tle_line2=tle_l2,
            minutes_step=5,
            total_hours=12,
        )
        real_data_source_note = "Downloaded current CelesTrak TLE and propagated with SGP4"
        print("Successfully created real-data-derived dense states from CelesTrak.")
    except Exception as e:
        print("CelesTrak download failed:", e)

# 3. Offline fallback only for code testing.
if real_dense_df is None:
    print("Using offline fallback TLE. Do not use this for final results unless clearly documented.")
    fallback_tle = """
ISS (ZARYA)
1 25544U 98067A   21073.51041667  .00002182  00000-0  51033-4 0  9993
2 25544  51.6442  21.3414 0002185  83.0492  48.0751 15.48915344273153
"""
    tle_name, tle_l1, tle_l2 = parse_tle_block(fallback_tle)
    real_dense_df, satrec_object = propagate_tle_to_states(
        tle_name=tle_name,
        tle_line1=tle_l1,
        tle_line2=tle_l2,
        minutes_step=5,
        total_hours=12,
    )
    real_data_source_note = "Offline fallback historical ISS TLE propagated with SGP4"

required_cols = ["t_sec", "x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms"]
missing = [c for c in required_cols if c not in real_dense_df.columns]
if missing:
    raise ValueError(f"Real-data table is missing required columns: {missing}")

real_dense_df = real_dense_df.dropna(subset=required_cols).copy().reset_index(drop=True)

print("Real-data source note:", real_data_source_note)
print("Rows:", len(real_dense_df))
display(real_dense_df.head())
display(real_dense_df[["t_sec", "altitude_km", "speed_kms"]].describe())

Downloading: https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=TLE


Successfully created real-data-derived dense states from CelesTrak.
Real-data source note: Downloaded current CelesTrak TLE and propagated with SGP4
Rows: 145


,timestamp_utc,error_code,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,dataset_name,object_name,source_name,t_sec,radius_km,speed_kms,altitude_km
0,2026-06-18 19:14:58.749487162+00:00,0,2558.580311,-6300.915910,0.006134,4.399827,1.792287,6.005514,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,0.0,6800.578984,7.657478,422.442684
1,2026-06-18 19:19:58.749487162+00:00,0,3708.767065,-5416.897510,1767.449001,3.194883,4.045089,5.664710,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,300.0,6798.647494,7.658914,420.511194
2,2026-06-18 19:24:58.749487162+00:00,0,4439.200507,-3919.803614,3334.285751,1.628275,5.840270,4.680973,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,600.0,6796.235942,7.659736,418.099642
3,2026-06-18 19:29:58.749487162+00:00,0,4667.286975,-1979.124907,4522.745187,-0.122083,6.974130,3.166419,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,900.0,6793.800640,7.660261,415.664340
4,2026-06-18 19:34:58.749487162+00:00,0,4367.389777,185.452079,5198.125233,-1.858123,7.318800,1.293124,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,1200.0,6791.832733,7.660916,413.696433


,t_sec,altitude_km,speed_kms
count,145.000000,145.000000,145.000000
mean,21600.000000,418.379630,7.658683
std,12600.595224,3.900878,0.004604
min,0.000000,412.404078,7.650721
25%,10800.000000,414.552466,7.654782
50%,21600.000000,418.809283,7.659636
75%,32400.000000,422.057317,7.662516
max,43200.000000,423.891924,7.665193


## 6. Create sparse observations from the dense real-data-derived trajectory

The PINN should not be given every dense state. Instead, it receives sparse observations. This simulates a realistic setting where the satellite is only observed occasionally.

For the first version, we add small measurement noise. The dense SGP4-derived trajectory is kept as a benchmark so we can calculate reconstruction error.

In [8]:
# Observation design
OBSERVATION_STRIDE = 6       # with 5-min dense steps, stride 6 means observations every 30 min
POSITION_NOISE_STD_KM = 0.20 # 200 meters position noise
VELOCITY_NOISE_STD_KMS = 0.0002 # 0.2 m/s velocity noise

sparse_obs_df = real_dense_df.iloc[::OBSERVATION_STRIDE].copy().reset_index(drop=True)

rng = np.random.default_rng(SEED)
for col in ["x_km", "y_km", "z_km"]:
    sparse_obs_df[col] = sparse_obs_df[col] + rng.normal(0.0, POSITION_NOISE_STD_KM, size=len(sparse_obs_df))
for col in ["vx_kms", "vy_kms", "vz_kms"]:
    sparse_obs_df[col] = sparse_obs_df[col] + rng.normal(0.0, VELOCITY_NOISE_STD_KMS, size=len(sparse_obs_df))

sparse_obs_df["observation_type"] = "sparse_noisy_real_data_derived"

print("Dense rows:", len(real_dense_df))
print("Sparse observation rows:", len(sparse_obs_df))
display(sparse_obs_df.head())

Dense rows: 145
Sparse observation rows: 25


,timestamp_utc,error_code,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,dataset_name,object_name,source_name,t_sec,radius_km,speed_kms,altitude_km,observation_type
0,2026-06-18 19:14:58.749487162+00:00,0,2558.641254,-6300.986337,0.063957,4.399965,1.792212,6.005585,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,0.0,6800.578984,7.657478,422.442684,sparse_noisy_real_data_derived
1,2026-06-18 19:44:58.749487162+00:00,0,2375.070702,4209.195561,4770.190765,-4.528062,5.578548,-2.665602,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,1800.0,6790.574007,7.663346,412.437707,sparse_noisy_real_data_derived
2,2026-06-18 20:14:58.749487162+00:00,0,-4652.070966,2568.492126,-4240.224888,-0.400605,-6.723665,-3.639421,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,3600.0,6798.306614,7.655774,420.170314,sparse_noisy_real_data_derived
3,2026-06-18 20:44:58.749487162+00:00,0,1717.696573,-6499.337470,-1032.512958,4.876884,0.356793,5.891387,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,5400.0,6801.341548,7.656364,423.205248,sparse_noisy_real_data_derived
4,2026-06-18 21:14:58.749487162+00:00,0,3125.965221,3153.014733,5137.646850,-3.887960,6.413151,-1.571012,celestrak_tle_sgp4_dense,ISS (ZARYA),CelesTrak TLE propagated with SGP4,7200.0,6790.540378,7.662516,412.404078,sparse_noisy_real_data_derived


In [9]:
plt.figure(figsize=(7, 7))
plt.plot(real_dense_df["x_km"], real_dense_df["y_km"], label="Dense SGP4-derived path")
plt.scatter(sparse_obs_df["x_km"], sparse_obs_df["y_km"], s=25, label="Sparse noisy observations")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title("Real-data-derived trajectory and sparse observations")
plt.axis("equal")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(real_dense_df["t_sec"] / 3600, real_dense_df["altitude_km"], label="Dense altitude")
plt.scatter(sparse_obs_df["t_sec"] / 3600, sparse_obs_df["altitude_km"], s=25, label="Sparse observations")
plt.xlabel("Time since first state [hours]")
plt.ylabel("Altitude [km]")
plt.title("Altitude over time")
plt.legend()
plt.grid(True)
plt.show()

## 7. Save the real-data-derived dense and sparse tables

These files are useful because they make the project reproducible.

In [10]:
real_dense_path = DATA_DIR / "real_drag_dense_tle_states.csv"
real_sparse_path = DATA_DIR / "real_drag_sparse_observations.csv"
real_metadata_path = DATA_DIR / "real_drag_metadata.csv"

metadata_rows = [
    {"parameter": "target_catnr", "value": TARGET_CATNR, "unit": "NORAD catalog number", "note": "Target satellite catalog number"},
    {"parameter": "target_name", "value": TARGET_NAME, "unit": "text", "note": "Target satellite name"},
    {"parameter": "real_data_source_note", "value": real_data_source_note, "unit": "text", "note": "How dense states were obtained"},
    {"parameter": "satellite_mass_kg", "value": SATELLITE_MASS_KG, "unit": "kg", "note": "Model assumption"},
    {"parameter": "reference_area_m2", "value": REFERENCE_AREA_M2, "unit": "m^2", "note": "Model assumption"},
    {"parameter": "drag_coefficient", "value": DRAG_COEFFICIENT, "unit": "dimensionless", "note": "Model assumption"},
    {"parameter": "prior_ballistic_coeff", "value": PRIOR_BALLISTIC_COEFF, "unit": "m^2/kg", "note": "Cd*A/m"},
    {"parameter": "position_noise_std_km", "value": POSITION_NOISE_STD_KM, "unit": "km", "note": "Noise added to sparse observations"},
    {"parameter": "velocity_noise_std_kms", "value": VELOCITY_NOISE_STD_KMS, "unit": "km/s", "note": "Noise added to sparse observations"},
    {"parameter": "observation_stride", "value": OBSERVATION_STRIDE, "unit": "rows", "note": "Sparse sampling interval"},
]
metadata_df = pd.DataFrame(metadata_rows)

real_dense_df.to_csv(real_dense_path, index=False)
sparse_obs_df.to_csv(real_sparse_path, index=False)
metadata_df.to_csv(real_metadata_path, index=False)

print("Saved:", real_dense_path)
print("Saved:", real_sparse_path)
print("Saved:", real_metadata_path)
display(metadata_df)

Saved: /private/tmp/pinn_run/nb4/real_drag_dense_tle_states.csv
Saved: /private/tmp/pinn_run/nb4/real_drag_sparse_observations.csv
Saved: /private/tmp/pinn_run/nb4/real_drag_metadata.csv


,parameter,value,unit,note
0,target_catnr,25544,NORAD catalog number,Target satellite catalog number
1,target_name,ISS (ZARYA),text,Target satellite name
2,real_data_source_note,Downloaded current CelesTrak TLE and propagate...,text,How dense states were obtained
3,satellite_mass_kg,420000.0,kg,Model assumption
4,reference_area_m2,1000.0,m^2,Model assumption
5,drag_coefficient,2.2,dimensionless,Model assumption
6,prior_ballistic_coeff,0.005238,m^2/kg,Cd*A/m
7,position_noise_std_km,0.2,km,Noise added to sparse observations
8,velocity_noise_std_kms,0.0002,km/s,Noise added to sparse observations
9,observation_stride,6,rows,Sparse sampling interval


## 8. Build the physical atmospheric drag model

This section defines the physical equations used by the PINN loss. The model includes:

1. two-body gravity,
2. \(J_2\) perturbation from Earth's oblateness,
3. Earth rotation,
4. relative velocity with respect to the atmosphere,
5. altitude-dependent atmospheric density,
6. drag acceleration using \(C_D A / m\).

The density model here is intentionally simple. It uses an exponential atmosphere centered near the observed orbit altitude. This is not an operational space-weather atmosphere model. However, it is differentiable, transparent, and appropriate for a first real-data PINN.

A future extension could replace this with NRLMSISE-00 or JB2008 and include solar flux and geomagnetic activity.

In [11]:
# SI units are used for the physics loss.
MU_EARTH = 3.986004418e14       # m^3 / s^2
R_EARTH = 6378136.3             # m
J2_EARTH = 1.08262668e-3
OMEGA_EARTH = 7.2921150e-5      # rad / s

# Density reference built around the target orbit.
# This is an educational exponential model. Density varies strongly with solar activity.
initial_altitude_m = float(real_dense_df["altitude_km"].iloc[0] * 1000.0)
H_SCALE_M = 60_000.0

# Approximate density near ISS-like altitude.
# The trainable density scale can absorb mismatch, but by default we keep it fixed
# because density and ballistic coefficient are strongly degenerate.
RHO_REF_KG_M3 = 3.5e-12
H_REF_M = initial_altitude_m

print("Initial altitude [km]:", initial_altitude_m / 1000)
print("Density reference altitude [km]:", H_REF_M / 1000)
print("Reference density [kg/m^3]:", RHO_REF_KG_M3)
print("Scale height [km]:", H_SCALE_M / 1000)

Initial altitude [km]: 422.4426838241834
Density reference altitude [km]: 422.4426838241834
Reference density [kg/m^3]: 3.5e-12
Scale height [km]: 60.0


In [12]:
def to_si_state_array(df):
    """Convert km and km/s columns into SI state columns: x,y,z in m and vx,vy,vz in m/s."""
    arr = df[["x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms"]].to_numpy(dtype=float).copy()
    arr[:, 0:3] *= 1000.0
    arr[:, 3:6] *= 1000.0
    return arr


def from_si_state_array(t_sec, arr_si):
    """Convert SI states back into a dataframe using km and km/s."""
    out = pd.DataFrame({
        "t_sec": t_sec,
        "x_km": arr_si[:, 0] / 1000.0,
        "y_km": arr_si[:, 1] / 1000.0,
        "z_km": arr_si[:, 2] / 1000.0,
        "vx_kms": arr_si[:, 3] / 1000.0,
        "vy_kms": arr_si[:, 4] / 1000.0,
        "vz_kms": arr_si[:, 5] / 1000.0,
    })
    out["radius_km"] = np.linalg.norm(out[["x_km", "y_km", "z_km"]].to_numpy(), axis=1)
    out["speed_kms"] = np.linalg.norm(out[["vx_kms", "vy_kms", "vz_kms"]].to_numpy(), axis=1)
    out["altitude_km"] = out["radius_km"] - R_EARTH / 1000.0
    return out


def density_exponential_np(altitude_m, rho_scale=1.0):
    """Simple exponential density model for plotting and classical baselines."""
    altitude_m = np.maximum(altitude_m, 120_000.0)
    rho = RHO_REF_KG_M3 * np.exp(-(altitude_m - H_REF_M) / H_SCALE_M)
    return rho_scale * rho


def rhs_numpy(t, state, ballistic_coeff=PRIOR_BALLISTIC_COEFF, rho_scale=1.0, use_j2=True, use_drag=True):
    """Classical orbital RHS in SI units for optional numerical baseline."""
    r = state[:3]
    v = state[3:]
    r_norm = np.linalg.norm(r)

    a_grav = -MU_EARTH * r / r_norm**3
    a_total = a_grav.copy()

    if use_j2:
        x, y, z = r
        r2 = r_norm**2
        z2 = z**2
        factor = 1.5 * J2_EARTH * MU_EARTH * R_EARTH**2 / r_norm**5
        common = 5.0 * z2 / r2
        a_j2 = np.array([
            factor * x * (common - 1.0),
            factor * y * (common - 1.0),
            factor * z * (common - 3.0),
        ])
        a_total = a_total + a_j2

    if use_drag:
        omega_vec = np.array([0.0, 0.0, OMEGA_EARTH])
        v_atm = np.cross(omega_vec, r)
        v_rel = v - v_atm
        v_rel_norm = np.linalg.norm(v_rel) + 1e-12
        altitude_m = r_norm - R_EARTH
        rho = density_exponential_np(altitude_m, rho_scale=rho_scale)
        a_drag = -0.5 * rho * ballistic_coeff * v_rel_norm * v_rel
        a_total = a_total + a_drag

    return np.concatenate([v, a_total])

## 9. Prepare tensors for PINN training

PINNs are sensitive to units and scaling. The raw positions are millions of meters, velocities are thousands of meters per second, and time spans thousands of seconds. We normalize the neural-network inputs and outputs, but compute the physics residual in physical SI units.

In [13]:
if not TORCH_AVAILABLE:
    raise ImportError("PyTorch is required for the PINN training sections.")

# Dense benchmark arrays
t_dense = real_dense_df["t_sec"].to_numpy(dtype=float)
y_dense_si = to_si_state_array(real_dense_df)

# Sparse observation arrays
t_obs = sparse_obs_df["t_sec"].to_numpy(dtype=float)
y_obs_si = to_si_state_array(sparse_obs_df)

# Normalize time to [0, 1]
T0 = float(t_dense.min())
T_SCALE = float(t_dense.max() - t_dense.min())
if T_SCALE <= 0:
    raise ValueError("Time span must be positive.")

# Normalize state using dense benchmark statistics for stable training.
# In a stricter experiment, use sparse observations only to define these scales.
STATE_MEAN = y_dense_si.mean(axis=0)
STATE_SCALE = y_dense_si.std(axis=0)
STATE_SCALE[STATE_SCALE < 1e-9] = 1.0


def normalize_time_np(t_sec):
    return ((np.asarray(t_sec) - T0) / T_SCALE).reshape(-1, 1)


def normalize_state_np(y_si):
    return (np.asarray(y_si) - STATE_MEAN) / STATE_SCALE


def denormalize_state_np(y_norm):
    return np.asarray(y_norm) * STATE_SCALE + STATE_MEAN


t_obs_norm = normalize_time_np(t_obs)
y_obs_norm = normalize_state_np(y_obs_si)

t_dense_norm = normalize_time_np(t_dense)
y_dense_norm = normalize_state_np(y_dense_si)

# Torch tensors
t_obs_tensor = torch.tensor(t_obs_norm, dtype=torch.float32, device=DEVICE)
y_obs_tensor = torch.tensor(y_obs_norm, dtype=torch.float32, device=DEVICE)

t_dense_tensor = torch.tensor(t_dense_norm, dtype=torch.float32, device=DEVICE)
y_dense_tensor = torch.tensor(y_dense_norm, dtype=torch.float32, device=DEVICE)

state_mean_tensor = torch.tensor(STATE_MEAN, dtype=torch.float32, device=DEVICE).reshape(1, 6)
state_scale_tensor = torch.tensor(STATE_SCALE, dtype=torch.float32, device=DEVICE).reshape(1, 6)

initial_state_norm_tensor = torch.tensor(y_obs_norm[0:1], dtype=torch.float32, device=DEVICE)

print("Dense benchmark shape:", y_dense_si.shape)
print("Sparse observation shape:", y_obs_si.shape)
print("T_SCALE [s]:", T_SCALE)
print("STATE_MEAN:", STATE_MEAN)
print("STATE_SCALE:", STATE_SCALE)

Dense benchmark shape: (145, 6)
Sparse observation shape: (25, 6)
T_SCALE [s]: 43200.0
STATE_MEAN: [ 2.47156913e+04  1.28783372e+05  8.25380124e+04 -1.23938573e+02
  9.07321651e+01 -1.00399628e+02]
STATE_SCALE: [3.30117179e+06 4.59183852e+06 3.76641993e+06 3.64979112e+03
 5.22282010e+03 4.24534362e+03]


## 10. Define differentiable Torch physics

The key PINN requirement is that the physics residual must be differentiable. These Torch functions mirror the NumPy functions above but operate entirely on Torch tensors.

In [14]:
def denormalize_state_torch(y_norm):
    return y_norm * state_scale_tensor + state_mean_tensor


def density_exponential_torch(altitude_m, log_rho_scale=None):
    """Differentiable exponential atmospheric density model."""
    altitude_m = torch.clamp(altitude_m, min=120_000.0)
    rho = RHO_REF_KG_M3 * torch.exp(-(altitude_m - H_REF_M) / H_SCALE_M)
    if log_rho_scale is not None:
        rho = rho * torch.exp(log_rho_scale)
    return rho


def acceleration_torch(r, v, log_ballistic_coeff, log_rho_scale=None, use_j2=True, use_drag=True):
    """Compute total acceleration from gravity, J2, and atmospheric drag in SI units."""
    r_norm = torch.linalg.norm(r, dim=1, keepdim=True)
    a_total = -MU_EARTH * r / (r_norm**3 + 1e-12)

    if use_j2:
        x = r[:, 0:1]
        y = r[:, 1:2]
        z = r[:, 2:3]
        r2 = r_norm**2
        z2 = z**2
        factor = 1.5 * J2_EARTH * MU_EARTH * R_EARTH**2 / (r_norm**5 + 1e-12)
        common = 5.0 * z2 / (r2 + 1e-12)
        a_j2 = torch.cat([
            factor * x * (common - 1.0),
            factor * y * (common - 1.0),
            factor * z * (common - 3.0),
        ], dim=1)
        a_total = a_total + a_j2

    if use_drag:
        omega_vec = torch.tensor([0.0, 0.0, OMEGA_EARTH], dtype=torch.float32, device=DEVICE).reshape(1, 3)
        omega_vec = omega_vec.repeat(r.shape[0], 1)
        v_atm = torch.cross(omega_vec, r, dim=1)
        v_rel = v - v_atm
        v_rel_norm = torch.linalg.norm(v_rel, dim=1, keepdim=True) + 1e-12
        altitude_m = r_norm - R_EARTH
        rho = density_exponential_torch(altitude_m, log_rho_scale=log_rho_scale)
        ballistic_coeff = torch.exp(log_ballistic_coeff)
        a_drag = -0.5 * rho * ballistic_coeff * v_rel_norm * v_rel
        a_total = a_total + a_drag

    return a_total

## 11. Build the PINN architecture

The neural network maps normalized time to normalized state:

\[
\hat{\mathbf{y}}(t) = [x, y, z, v_x, v_y, v_z]
\]

To make the initial condition easier to satisfy, the model is written as:

\[
\hat{\mathbf{y}}(t) = \mathbf{y}_0 + t \cdot NN(t)
\]

This forces the model output to equal the initial state at \(t = 0\) before any training.

In [15]:
class MLP(nn.Module):
    def __init__(self, input_dim=1, output_dim=6, hidden_width=96, hidden_depth=4, activation="tanh"):
        super().__init__()
        if activation == "tanh":
            act = nn.Tanh
        elif activation == "silu":
            act = nn.SiLU
        elif activation == "relu":
            act = nn.ReLU
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        layers = []
        in_dim = input_dim
        for _ in range(hidden_depth):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(act())
            in_dim = hidden_width
        layers.append(nn.Linear(hidden_width, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)


class InitialConditionPINN(nn.Module):
    def __init__(self, initial_state_norm, hidden_width=96, hidden_depth=4, activation="tanh"):
        super().__init__()
        self.register_buffer("initial_state_norm", initial_state_norm.clone().detach())
        self.net = MLP(
            input_dim=1,
            output_dim=6,
            hidden_width=hidden_width,
            hidden_depth=hidden_depth,
            activation=activation,
        )

    def forward(self, t):
        return self.initial_state_norm + t * self.net(t)


def make_model_and_parameters(
    hidden_width=96,
    hidden_depth=4,
    activation="tanh",
    initial_ballistic_coeff=PRIOR_BALLISTIC_COEFF,
    learn_density_scale=False,
):
    model = InitialConditionPINN(
        initial_state_norm=initial_state_norm_tensor,
        hidden_width=hidden_width,
        hidden_depth=hidden_depth,
        activation=activation,
    ).to(DEVICE)

    # Learn log(B) so B remains positive.
    log_ballistic_coeff = nn.Parameter(
        torch.tensor(math.log(initial_ballistic_coeff), dtype=torch.float32, device=DEVICE)
    )

    # Optional density-scale learning. Keep disabled by default because density and B are degenerate.
    if learn_density_scale:
        log_rho_scale = nn.Parameter(torch.tensor(0.0, dtype=torch.float32, device=DEVICE))
    else:
        log_rho_scale = torch.tensor(0.0, dtype=torch.float32, device=DEVICE)

    return model, log_ballistic_coeff, log_rho_scale

## 12. Define the PINN loss function

The total loss has four parts:

1. **Data loss:** match sparse real-data-derived observations.
2. **Physics loss:** satisfy the orbital equations at collocation points.
3. **Initial condition loss:** keep the first state anchored.
4. **Parameter prior loss:** prevent the learned ballistic coefficient from drifting into an unphysical range.

The physics residual is:

\[
\frac{d\mathbf{r}}{dt} - \mathbf{v} = 0
\]

\[
\frac{d\mathbf{v}}{dt} - \left(\mathbf{a}_{gravity} + \mathbf{a}_{J_2} + \mathbf{a}_{drag}\right) = 0
\]

In [16]:
def compute_pinn_losses(
    model,
    log_ballistic_coeff,
    log_rho_scale,
    t_obs_tensor,
    y_obs_tensor,
    t_colloc_tensor,
    weights,
    use_j2=True,
    use_drag=True,
    learn_density_scale=False,
):
    # -------------------------
    # 1. Data loss
    # -------------------------
    pred_obs_norm = model(t_obs_tensor)
    data_loss = torch.mean((pred_obs_norm - y_obs_tensor) ** 2)

    # -------------------------
    # 2. Initial condition loss
    # -------------------------
    pred_initial_norm = model(torch.zeros((1, 1), dtype=torch.float32, device=DEVICE))
    ic_loss = torch.mean((pred_initial_norm - initial_state_norm_tensor) ** 2)

    # -------------------------
    # 3. Physics residual loss
    # -------------------------
    t_col = t_colloc_tensor.clone().detach().requires_grad_(True)
    pred_col_norm = model(t_col)
    pred_col_si = denormalize_state_torch(pred_col_norm)

    grads = []
    for state_index in range(6):
        grad_i = torch.autograd.grad(
            outputs=pred_col_si[:, state_index:state_index + 1],
            inputs=t_col,
            grad_outputs=torch.ones_like(pred_col_si[:, state_index:state_index + 1]),
            create_graph=True,
            retain_graph=True,
        )[0]
        grads.append(grad_i)

    # dy/dt_norm_time to dy/dt_seconds
    dy_dt_si = torch.cat(grads, dim=1) / T_SCALE

    r = pred_col_si[:, 0:3]
    v = pred_col_si[:, 3:6]
    expected_acc = acceleration_torch(
        r=r,
        v=v,
        log_ballistic_coeff=log_ballistic_coeff,
        log_rho_scale=log_rho_scale if learn_density_scale else None,
        use_j2=use_j2,
        use_drag=use_drag,
    )

    residual_pos = dy_dt_si[:, 0:3] - v
    residual_vel = dy_dt_si[:, 3:6] - expected_acc

    # Normalize residuals so position and velocity residuals contribute comparable scales.
    # Typical velocity scale is ~7600 m/s. Typical acceleration scale is ~8 m/s^2.
    pos_residual_loss = torch.mean((residual_pos / 7600.0) ** 2)
    vel_residual_loss = torch.mean((residual_vel / 8.7) ** 2)
    physics_loss = pos_residual_loss + vel_residual_loss

    # -------------------------
    # 4. Parameter prior loss
    # -------------------------
    log_prior_B = math.log(PRIOR_BALLISTIC_COEFF)
    prior_B_loss = (log_ballistic_coeff - log_prior_B) ** 2

    if learn_density_scale:
        prior_rho_loss = log_rho_scale ** 2
    else:
        prior_rho_loss = torch.tensor(0.0, dtype=torch.float32, device=DEVICE)

    prior_loss = prior_B_loss + 0.25 * prior_rho_loss

    total = (
        weights["data"] * data_loss
        + weights["ic"] * ic_loss
        + weights["physics"] * physics_loss
        + weights["prior"] * prior_loss
    )

    return {
        "total": total,
        "data": data_loss,
        "ic": ic_loss,
        "physics": physics_loss,
        "pos_residual": pos_residual_loss,
        "vel_residual": vel_residual_loss,
        "prior": prior_loss,
    }

## 13. Train the real-data atmospheric drag PINN

For a first live session, keep the number of epochs modest. Once the notebook runs successfully, increase `epochs`, `n_collocation`, and possibly `hidden_width`.

Recommended live-session path:

1. Run 300 to 500 epochs first.
2. Confirm losses are decreasing.
3. Check the reconstruction plot.
4. Then increase to 1500 to 3000 epochs for a stronger final result.

In [17]:
def train_real_drag_pinn(
    hidden_width=96,
    hidden_depth=4,
    activation="tanh",
    learning_rate=1e-3,
    n_collocation=180,
    epochs=500,
    weights=None,
    learn_density_scale=False,
    print_every=50,
):
    if weights is None:
        weights = {"data": 20.0, "ic": 10.0, "physics": 1.0, "prior": 0.05}

    model, log_B, log_rho_scale = make_model_and_parameters(
        hidden_width=hidden_width,
        hidden_depth=hidden_depth,
        activation=activation,
        initial_ballistic_coeff=PRIOR_BALLISTIC_COEFF,
        learn_density_scale=learn_density_scale,
    )

    params = list(model.parameters()) + [log_B]
    if learn_density_scale and isinstance(log_rho_scale, nn.Parameter):
        params.append(log_rho_scale)

    optimizer = torch.optim.Adam(params, lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(epochs // 3, 1), gamma=0.5)

    t_colloc = torch.linspace(0.0, 1.0, n_collocation, dtype=torch.float32, device=DEVICE).reshape(-1, 1)

    history = []
    start = time.time()

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        losses = compute_pinn_losses(
            model=model,
            log_ballistic_coeff=log_B,
            log_rho_scale=log_rho_scale,
            t_obs_tensor=t_obs_tensor,
            y_obs_tensor=y_obs_tensor,
            t_colloc_tensor=t_colloc,
            weights=weights,
            use_j2=True,
            use_drag=True,
            learn_density_scale=learn_density_scale,
        )

        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if epoch == 1 or epoch % print_every == 0 or epoch == epochs:
            learned_B = float(torch.exp(log_B).detach().cpu())
            if learn_density_scale:
                learned_rho_scale = float(torch.exp(log_rho_scale).detach().cpu())
            else:
                learned_rho_scale = 1.0

            row = {
                "epoch": epoch,
                "total": float(losses["total"].detach().cpu()),
                "data": float(losses["data"].detach().cpu()),
                "ic": float(losses["ic"].detach().cpu()),
                "physics": float(losses["physics"].detach().cpu()),
                "pos_residual": float(losses["pos_residual"].detach().cpu()),
                "vel_residual": float(losses["vel_residual"].detach().cpu()),
                "prior": float(losses["prior"].detach().cpu()),
                "learned_B_m2_per_kg": learned_B,
                "learned_rho_scale": learned_rho_scale,
                "learning_rate": scheduler.get_last_lr()[0],
            }
            history.append(row)
            print(
                f"Epoch {epoch:5d} | total={row['total']:.3e} | data={row['data']:.3e} | "
                f"physics={row['physics']:.3e} | B={learned_B:.3e} | rho_scale={learned_rho_scale:.3e}"
            )

    elapsed = time.time() - start
    print(f"Training completed in {elapsed:.1f} seconds.")
    return model, log_B, log_rho_scale, pd.DataFrame(history)

In [18]:
BASELINE_CONFIG = {
    "hidden_width": 96,
    "hidden_depth": 4,
    "activation": "tanh",
    "learning_rate": 1e-3,
    "n_collocation": 160,
    "epochs": 500,
    "weights": {"data": 20.0, "ic": 10.0, "physics": 1.0, "prior": 0.05},
    "learn_density_scale": False,
    "print_every": 50,
}

model, log_B, log_rho_scale, history_df = train_real_drag_pinn(**BASELINE_CONFIG)

learned_B = float(torch.exp(log_B).detach().cpu())
learned_rho_scale = float(torch.exp(log_rho_scale).detach().cpu()) if isinstance(log_rho_scale, nn.Parameter) else 1.0

print("\nPrior B = Cd*A/m [m^2/kg]:", PRIOR_BALLISTIC_COEFF)
print("Learned B [m^2/kg]:", learned_B)
print("Learned B / prior B:", learned_B / PRIOR_BALLISTIC_COEFF)
print("Learned density scale:", learned_rho_scale)

display(history_df.tail())

Epoch     1 | total=4.119e+01 | data=2.023e+00 | physics=7.294e-01 | B=5.236e-03 | rho_scale=1.000e+00


Epoch    50 | total=2.655e+01 | data=1.250e+00 | physics=1.536e+00 | B=5.238e-03 | rho_scale=1.000e+00


Epoch   100 | total=2.519e+01 | data=1.179e+00 | physics=1.608e+00 | B=5.238e-03 | rho_scale=1.000e+00


Epoch   150 | total=2.502e+01 | data=1.175e+00 | physics=1.510e+00 | B=5.238e-03 | rho_scale=1.000e+00


Epoch   200 | total=2.490e+01 | data=1.164e+00 | physics=1.618e+00 | B=5.238e-03 | rho_scale=1.000e+00


...

Epoch   350 | total=2.421e+01 | data=1.131e+00 | physics=1.594e+00 | B=5.238e-03 | rho_scale=1.000e+00


Epoch   400 | total=2.394e+01 | data=1.117e+00 | physics=1.595e+00 | B=5.238e-03 | rho_scale=1.000e+00


Epoch   450 | total=2.348e+01 | data=1.094e+00 | physics=1.591e+00 | B=5.238e-03 | rho_scale=1.000e+00


Epoch   500 | total=2.259e+01 | data=1.051e+00 | physics=1.572e+00 | B=5.238e-03 | rho_scale=1.000e+00
Training completed in 10.7 seconds.

Prior B = Cd*A/m [m^2/kg]: 0.005238095238095238
Learned B [m^2/kg]: 0.005238098558038473
Learned B / prior B: 1.0000006338073448
Learned density scale: 1.0


,epoch,total,data,ic,physics,pos_residual,vel_residual,prior,learned_B_m2_per_kg,learned_rho_scale,learning_rate
6,300,24.476820,1.145806,0.0,1.560694,0.020596,1.540098,1.760782e-09,0.005238,1.0,0.000500
7,350,24.212881,1.130967,0.0,1.593537,0.023714,1.569823,4.456524e-11,0.005238,1.0,0.000250
8,400,23.936611,1.117103,0.0,1.594542,0.026608,1.567934,2.273737e-13,0.005238,1.0,0.000250
9,450,23.479357,1.094442,0.0,1.590518,0.025524,1.564993,2.273737e-13,0.005238,1.0,0.000250
10,500,22.586208,1.050716,0.0,1.571880,0.024994,1.546886,2.273737e-13,0.005238,1.0,0.000125


## 14. Evaluate reconstruction quality

The sparse observations are what the PINN is trained on. The dense SGP4-derived trajectory is used only as a benchmark for evaluation.

Useful metrics:

- position RMSE in km,
- velocity RMSE in m/s,
- final position error in km,
- learned \(C_DA/m\) relative to the prior assumption.

In [19]:
def predict_pinn_dataframe(model, t_sec_values):
    model.eval()
    with torch.no_grad():
        t_norm = torch.tensor(normalize_time_np(t_sec_values), dtype=torch.float32, device=DEVICE)
        pred_norm = model(t_norm).detach().cpu().numpy()
    pred_si = denormalize_state_np(pred_norm)
    return from_si_state_array(np.asarray(t_sec_values), pred_si)

pred_dense_df = predict_pinn_dataframe(model, t_dense)

truth_pos_km = real_dense_df[["x_km", "y_km", "z_km"]].to_numpy()
pred_pos_km = pred_dense_df[["x_km", "y_km", "z_km"]].to_numpy()
truth_vel_ms = real_dense_df[["vx_kms", "vy_kms", "vz_kms"]].to_numpy() * 1000.0
pred_vel_ms = pred_dense_df[["vx_kms", "vy_kms", "vz_kms"]].to_numpy() * 1000.0

position_error_km = np.linalg.norm(pred_pos_km - truth_pos_km, axis=1)
velocity_error_ms = np.linalg.norm(pred_vel_ms - truth_vel_ms, axis=1)

summary_df = pd.DataFrame({
    "metric": [
        "position_rmse_km",
        "position_mae_km",
        "final_position_error_km",
        "velocity_rmse_m_per_s",
        "velocity_mae_m_per_s",
        "prior_ballistic_coeff_m2_per_kg",
        "learned_ballistic_coeff_m2_per_kg",
        "learned_B_over_prior_B",
        "learned_density_scale",
    ],
    "value": [
        float(np.sqrt(np.mean(position_error_km**2))),
        float(np.mean(position_error_km)),
        float(position_error_km[-1]),
        float(np.sqrt(np.mean(velocity_error_ms**2))),
        float(np.mean(velocity_error_ms)),
        float(PRIOR_BALLISTIC_COEFF),
        float(learned_B),
        float(learned_B / PRIOR_BALLISTIC_COEFF),
        float(learned_rho_scale),
    ],
})

display(summary_df)

,metric,value
0,position_rmse_km,7944.313074
1,position_mae_km,7365.494822
2,final_position_error_km,2615.781870
3,velocity_rmse_m_per_s,7367.503979
4,velocity_mae_m_per_s,7151.220397
5,prior_ballistic_coeff_m2_per_kg,0.005238
6,learned_ballistic_coeff_m2_per_kg,0.005238
7,learned_B_over_prior_B,1.000001
8,learned_density_scale,1.000000


In [20]:
plt.figure(figsize=(8, 5))
for col in ["total", "data", "physics", "prior"]:
    plt.semilogy(history_df["epoch"], history_df[col], label=col)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Real-data atmospheric drag PINN training losses")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 7))
plt.plot(real_dense_df["x_km"], real_dense_df["y_km"], label="Dense SGP4-derived benchmark")
plt.scatter(sparse_obs_df["x_km"], sparse_obs_df["y_km"], s=25, label="Sparse observations")
plt.plot(pred_dense_df["x_km"], pred_dense_df["y_km"], label="PINN reconstruction")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title("Trajectory reconstruction in the xy-plane")
plt.axis("equal")
plt.legend()
plt.grid(True)
plt.show()

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot(real_dense_df["x_km"], real_dense_df["y_km"], real_dense_df["z_km"], label="Dense benchmark")
ax.scatter(sparse_obs_df["x_km"], sparse_obs_df["y_km"], sparse_obs_df["z_km"], s=20, label="Sparse obs")
ax.plot(pred_dense_df["x_km"], pred_dense_df["y_km"], pred_dense_df["z_km"], label="PINN reconstruction")
ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_zlabel("z [km]")
ax.set_title("3D trajectory reconstruction")
ax.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t_dense / 3600.0, position_error_km)
plt.xlabel("Time since first state [hours]")
plt.ylabel("Position error [km]")
plt.title("PINN position reconstruction error")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t_dense / 3600.0, velocity_error_ms)
plt.xlabel("Time since first state [hours]")
plt.ylabel("Velocity error [m/s]")
plt.title("PINN velocity reconstruction error")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(real_dense_df["t_sec"] / 3600.0, real_dense_df["altitude_km"], label="Dense benchmark")
plt.plot(pred_dense_df["t_sec"] / 3600.0, pred_dense_df["altitude_km"], label="PINN reconstruction")
plt.scatter(sparse_obs_df["t_sec"] / 3600.0, sparse_obs_df["altitude_km"], s=20, label="Sparse observations")
plt.xlabel("Time since first state [hours]")
plt.ylabel("Altitude [km]")
plt.title("Altitude reconstruction")
plt.legend()
plt.grid(True)
plt.show()

## 15. Optional classical baseline with the same drag model

This section numerically integrates the first observed state forward using the same physical model. This creates a baseline that is not a neural network.

This is useful for the final paper because it lets the student compare:

- sparse observations,
- a classical ODE propagation baseline,
- the PINN reconstruction.

The baseline is optional because the PINN training is the main focus of this notebook.

In [21]:
RUN_CLASSICAL_BASELINE = True

if RUN_CLASSICAL_BASELINE and SCIPY_AVAILABLE:
    y0_si = y_obs_si[0]
    sol = solve_ivp(
        fun=lambda t, y: rhs_numpy(
            t,
            y,
            ballistic_coeff=PRIOR_BALLISTIC_COEFF,
            rho_scale=1.0,
            use_j2=True,
            use_drag=True,
        ),
        t_span=(float(t_dense[0]), float(t_dense[-1])),
        y0=y0_si,
        t_eval=t_dense,
        rtol=1e-8,
        atol=1e-8,
        method="RK45",
    )

    if sol.success:
        baseline_df = from_si_state_array(t_dense, sol.y.T)
        baseline_pos_km = baseline_df[["x_km", "y_km", "z_km"]].to_numpy()
        baseline_error_km = np.linalg.norm(baseline_pos_km - truth_pos_km, axis=1)

        baseline_summary = pd.DataFrame({
            "metric": ["baseline_position_rmse_km", "baseline_final_position_error_km"],
            "value": [
                float(np.sqrt(np.mean(baseline_error_km**2))),
                float(baseline_error_km[-1]),
            ],
        })
        display(baseline_summary)

        plt.figure(figsize=(7, 7))
        plt.plot(real_dense_df["x_km"], real_dense_df["y_km"], label="Dense benchmark")
        plt.plot(baseline_df["x_km"], baseline_df["y_km"], label="Classical drag baseline")
        plt.plot(pred_dense_df["x_km"], pred_dense_df["y_km"], label="PINN reconstruction")
        plt.xlabel("x [km]")
        plt.ylabel("y [km]")
        plt.title("Classical baseline vs PINN")
        plt.axis("equal")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("Classical integration failed:", sol.message)
elif RUN_CLASSICAL_BASELINE and not SCIPY_AVAILABLE:
    print("SciPy is not available, so the classical baseline was skipped.")
else:
    print("Classical baseline skipped.")

,metric,value
0,baseline_position_rmse_km,16.571491
1,baseline_final_position_error_km,28.553865


## 16. Optional hyperparameter tuning

Turn this on after the baseline notebook runs successfully. For a live session, leave it off at first.

Suggested tuning questions:

- Does a wider network reduce reconstruction error?
- Does a larger physics weight improve physical consistency or make data fitting worse?
- Does learning a density scale create non-identifiability with \(C_D A / m\)?

In [22]:
RUN_HYPERPARAMETER_SWEEP = False

if RUN_HYPERPARAMETER_SWEEP:
    sweep_configs = [
        {
            "name": "baseline",
            "hidden_width": 96,
            "hidden_depth": 4,
            "learning_rate": 1e-3,
            "n_collocation": 160,
            "epochs": 350,
            "weights": {"data": 20.0, "ic": 10.0, "physics": 1.0, "prior": 0.05},
            "learn_density_scale": False,
            "print_every": 175,
        },
        {
            "name": "higher_physics_weight",
            "hidden_width": 96,
            "hidden_depth": 4,
            "learning_rate": 1e-3,
            "n_collocation": 180,
            "epochs": 350,
            "weights": {"data": 20.0, "ic": 10.0, "physics": 5.0, "prior": 0.05},
            "learn_density_scale": False,
            "print_every": 175,
        },
        {
            "name": "wider_network",
            "hidden_width": 128,
            "hidden_depth": 4,
            "learning_rate": 8e-4,
            "n_collocation": 200,
            "epochs": 350,
            "weights": {"data": 20.0, "ic": 10.0, "physics": 1.0, "prior": 0.05},
            "learn_density_scale": False,
            "print_every": 175,
        },
        {
            "name": "learn_density_scale_warning",
            "hidden_width": 96,
            "hidden_depth": 4,
            "learning_rate": 8e-4,
            "n_collocation": 180,
            "epochs": 350,
            "weights": {"data": 20.0, "ic": 10.0, "physics": 1.0, "prior": 0.1},
            "learn_density_scale": True,
            "print_every": 175,
        },
    ]

    sweep_rows = []

    for cfg in sweep_configs:
        print("\n" + "=" * 80)
        print("Running:", cfg["name"])
        print("=" * 80)
        run_cfg = cfg.copy()
        name = run_cfg.pop("name")
        sweep_model, sweep_log_B, sweep_log_rho, sweep_history = train_real_drag_pinn(**run_cfg)
        sweep_pred = predict_pinn_dataframe(sweep_model, t_dense)
        sweep_pos = sweep_pred[["x_km", "y_km", "z_km"]].to_numpy()
        sweep_vel = sweep_pred[["vx_kms", "vy_kms", "vz_kms"]].to_numpy() * 1000.0
        sweep_pos_err = np.linalg.norm(sweep_pos - truth_pos_km, axis=1)
        sweep_vel_err = np.linalg.norm(sweep_vel - truth_vel_ms, axis=1)

        sweep_rows.append({
            "name": name,
            "position_rmse_km": float(np.sqrt(np.mean(sweep_pos_err**2))),
            "velocity_rmse_m_per_s": float(np.sqrt(np.mean(sweep_vel_err**2))),
            "learned_B_m2_per_kg": float(torch.exp(sweep_log_B).detach().cpu()),
            "learned_B_over_prior": float(torch.exp(sweep_log_B).detach().cpu()) / PRIOR_BALLISTIC_COEFF,
            "learned_rho_scale": float(torch.exp(sweep_log_rho).detach().cpu()) if isinstance(sweep_log_rho, nn.Parameter) else 1.0,
            "final_total_loss": float(sweep_history["total"].iloc[-1]),
        })

    sweep_df = pd.DataFrame(sweep_rows).sort_values("position_rmse_km")
    display(sweep_df)
    sweep_df.to_csv(OUTPUT_DIR / "real_drag_pinn_hyperparameter_sweep.csv", index=False)
else:
    print("Hyperparameter sweep is turned off. Set RUN_HYPERPARAMETER_SWEEP = True to run it.")

Hyperparameter sweep is turned off. Set RUN_HYPERPARAMETER_SWEEP = True to run it.


## 17. Save final model outputs

These outputs are the most important files to preserve for the final paper:

- model weights,
- training history,
- dense reconstruction,
- sparse observations,
- summary metrics,
- metadata.

In [23]:
model_output_path = OUTPUT_DIR / "real_drag_pinn_model.pt"
history_output_path = OUTPUT_DIR / "real_drag_pinn_training_history.csv"
reconstruction_output_path = OUTPUT_DIR / "real_drag_pinn_reconstruction.csv"
summary_output_path = OUTPUT_DIR / "real_drag_pinn_summary.csv"
metadata_output_path = OUTPUT_DIR / "real_drag_pinn_metadata.json"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "log_ballistic_coeff": log_B.detach().cpu(),
    "log_rho_scale": log_rho_scale.detach().cpu() if hasattr(log_rho_scale, "detach") else torch.tensor(0.0),
    "config": BASELINE_CONFIG,
    "normalization": {
        "T0": T0,
        "T_SCALE": T_SCALE,
        "STATE_MEAN": STATE_MEAN.tolist(),
        "STATE_SCALE": STATE_SCALE.tolist(),
    },
    "physical_constants": {
        "MU_EARTH": MU_EARTH,
        "R_EARTH": R_EARTH,
        "J2_EARTH": J2_EARTH,
        "OMEGA_EARTH": OMEGA_EARTH,
        "RHO_REF_KG_M3": RHO_REF_KG_M3,
        "H_REF_M": H_REF_M,
        "H_SCALE_M": H_SCALE_M,
    },
    "satellite_assumptions": {
        "mass_kg": SATELLITE_MASS_KG,
        "reference_area_m2": REFERENCE_AREA_M2,
        "drag_coefficient": DRAG_COEFFICIENT,
        "prior_ballistic_coeff_m2_per_kg": PRIOR_BALLISTIC_COEFF,
    },
}

torch.save(checkpoint, model_output_path)
history_df.to_csv(history_output_path, index=False)
pred_dense_df.to_csv(reconstruction_output_path, index=False)
summary_df.to_csv(summary_output_path, index=False)

metadata_payload = {
    "project_title": "Characterizing Orbital Perturbations via Physics-Informed Neural Networks",
    "notebook": "Notebook 4: Real-Data Atmospheric Drag PINN",
    "real_data_source_note": real_data_source_note,
    "target_catnr": TARGET_CATNR,
    "target_name": TARGET_NAME,
    "satellite_assumptions": checkpoint["satellite_assumptions"],
    "physical_constants": checkpoint["physical_constants"],
    "scientific_caveat": "TLE/SGP4 states are real-data-derived benchmark states, not raw tracking measurements.",
}

with open(metadata_output_path, "w") as f:
    json.dump(metadata_payload, f, indent=2)

print("Saved model:", model_output_path)
print("Saved history:", history_output_path)
print("Saved reconstruction:", reconstruction_output_path)
print("Saved summary:", summary_output_path)
print("Saved metadata:", metadata_output_path)

Saved model: /private/tmp/pinn_run/nb4/pinn_outputs_real_drag/real_drag_pinn_model.pt
Saved history: /private/tmp/pinn_run/nb4/pinn_outputs_real_drag/real_drag_pinn_training_history.csv
Saved reconstruction: /private/tmp/pinn_run/nb4/pinn_outputs_real_drag/real_drag_pinn_reconstruction.csv
Saved summary: /private/tmp/pinn_run/nb4/pinn_outputs_real_drag/real_drag_pinn_summary.csv
Saved metadata: /private/tmp/pinn_run/nb4/pinn_outputs_real_drag/real_drag_pinn_metadata.json


## 18. How to explain this notebook in the paper

A good methods paragraph could say:

> We used public TLE data for a low-Earth-orbit satellite and propagated the TLE using SGP4 to create a dense real-data-derived Cartesian state-vector benchmark. We then downsampled this benchmark into sparse noisy observations and trained a Physics-Informed Neural Network to reconstruct the full state trajectory. Unlike the earlier simplified drag model, the physical residual in this notebook included two-body gravity, the Earth \(J_2\) perturbation, atmospheric co-rotation, an altitude-dependent exponential atmospheric density model, and a drag acceleration proportional to \(\rho C_D A/m\). The inverse parameter inferred by the PINN was the ballistic-coefficient-like quantity \(C_D A/m\), initialized using assumed satellite mass, reference area, and drag coefficient.

Important limitations to state clearly:

1. TLE/SGP4 output is not raw observational tracking data.
2. Satellite mass and reference area are assumptions unless a verified spacecraft-specific source is used.
3. Atmospheric density is highly variable and depends on solar and geomagnetic activity.
4. The product \(\rho C_D A/m\) is easier to infer than density, drag coefficient, area, and mass separately.
5. The learned ballistic coefficient should be interpreted as an effective parameter under this model, not as a definitive spacecraft property.

## 19. Next research upgrades

Strong next steps:

1. Replace the simple exponential density with NRLMSISE-00 or JB2008.
2. Use historical Space-Track TLE sequences instead of only one current TLE.
3. Compare ISS, Starlink, and Iridium satellites to test whether drag recovery behaves differently across altitude and area-to-mass regimes.
4. Add an ablation study: two-body only, two-body plus \(J_2\), and two-body plus \(J_2\) plus drag.
5. Add a standard neural-network baseline with no physics residual.
6. Add a classical numerical least-squares estimator for \(C_D A/m\) as a non-ML comparison.

In [24]:
## === IMPROVED MODEL: Fourier-feature PINN for the real ISS trajectory ===
# The plain tanh-MLP PINN above cannot represent the ~7-8 high-frequency orbital
# cycles in the 12-hour ISS window (spectral bias), so its reconstruction collapses.
# Embedding normalized time in sinusoidal Fourier features fixes this. We reuse the
# SI-unit differentiable physics (gravity + J2 + co-rotating exponential-atmosphere
# drag) defined above, and again learn log(B = Cd*A/m).
import math, json
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42); np.random.seed(42)

_freqsB = torch.arange(1, 25, dtype=torch.float32, device=DEVICE).reshape(1, -1)
def fourier_features_B(t):
    ang = 2 * math.pi * t * _freqsB
    return torch.cat([t, torch.sin(ang), torch.cos(ang)], dim=1)

class FourierPINN_B(nn.Module):
    def __init__(self, emb_dim, width=128, depth=3):
        super().__init__()
        layers = [nn.Linear(emb_dim, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 6)]
        self.net = nn.Sequential(*layers)
    def forward(self, emb):
        return self.net(emb)

_obs_embB = fourier_features_B(t_obs_tensor)
_emb0B = fourier_features_B(torch.zeros((1, 1), dtype=torch.float32, device=DEVICE))
_collocB = torch.linspace(0, 1, 200, device=DEVICE).reshape(-1, 1)

fpinnB = FourierPINN_B(_obs_embB.shape[1]).to(DEVICE)
log_B_fourier = nn.Parameter(torch.tensor(math.log(PRIOR_BALLISTIC_COEFF), dtype=torch.float32, device=DEVICE))
_optB = torch.optim.Adam(list(fpinnB.parameters()) + [log_B_fourier], lr=2e-3)
_schedB = torch.optim.lr_scheduler.StepLR(_optB, step_size=2000, gamma=0.5)

FOURIER_EPOCHS_B = 5000
WB = {"data": 20.0, "ic": 10.0, "physics": 1.0, "prior": 0.05}
fourierB_history = {"epoch": [], "total": [], "data": [], "physics": [], "B": []}
_logprior = math.log(PRIOR_BALLISTIC_COEFF)
for epoch in range(1, FOURIER_EPOCHS_B + 1):
    _optB.zero_grad()
    data_loss = torch.mean((fpinnB(_obs_embB) - y_obs_tensor) ** 2)
    ic_loss = torch.mean((fpinnB(_emb0B) - initial_state_norm_tensor) ** 2)
    tc = _collocB.clone().detach().requires_grad_(True)
    yc = denormalize_state_torch(fpinnB(fourier_features_B(tc)))
    grads = [torch.autograd.grad(yc[:, i:i+1], tc, torch.ones_like(yc[:, i:i+1]),
                                 create_graph=True, retain_graph=True)[0] for i in range(6)]
    dydt = torch.cat(grads, dim=1) / T_SCALE
    r, v = yc[:, 0:3], yc[:, 3:6]
    acc = acceleration_torch(r=r, v=v, log_ballistic_coeff=log_B_fourier,
                             log_rho_scale=None, use_j2=True, use_drag=True)
    physics_loss = torch.mean(((dydt[:, 0:3] - v) / 7600.0) ** 2) + torch.mean(((dydt[:, 3:6] - acc) / 8.7) ** 2)
    prior_loss = (log_B_fourier - _logprior) ** 2
    total = WB["data"] * data_loss + WB["ic"] * ic_loss + WB["physics"] * physics_loss + WB["prior"] * prior_loss
    total.backward()
    torch.nn.utils.clip_grad_norm_(list(fpinnB.parameters()) + [log_B_fourier], 1.0)
    _optB.step(); _schedB.step()
    if epoch % 500 == 0 or epoch == 1:
        Bv = float(torch.exp(log_B_fourier).detach().cpu())
        fourierB_history["epoch"].append(epoch)
        fourierB_history["total"].append(float(total.detach().cpu()))
        fourierB_history["data"].append(float(data_loss.detach().cpu()))
        fourierB_history["physics"].append(float(physics_loss.detach().cpu()))
        fourierB_history["B"].append(Bv)
        print(f"Epoch {epoch:5d} | total={float(total):.3e} | data={float(data_loss):.3e} | "
              f"physics={float(physics_loss):.3e} | B={Bv:.4e}")

fpinnB.eval()
with torch.no_grad():
    _tn = torch.tensor(normalize_time_np(t_dense), dtype=torch.float32, device=DEVICE)
    _predn = fpinnB(fourier_features_B(_tn)).cpu().numpy()
_pred_si = denormalize_state_np(_predn)
_pred_pos_km = _pred_si[:, 0:3] / 1000.0
_pred_vel_ms = _pred_si[:, 3:6]
_truth_pos_km = real_dense_df[["x_km", "y_km", "z_km"]].to_numpy()
_truth_vel_ms = real_dense_df[["vx_kms", "vy_kms", "vz_kms"]].to_numpy() * 1000.0
fourierB_pos_rmse = float(np.sqrt(np.mean(np.sum((_pred_pos_km - _truth_pos_km) ** 2, axis=1))))
fourierB_vel_rmse = float(np.sqrt(np.mean(np.sum((_pred_vel_ms - _truth_vel_ms) ** 2, axis=1))))
fourierB_final_pos_err = float(np.linalg.norm(_pred_pos_km[-1] - _truth_pos_km[-1]))
fourierB_learned = float(torch.exp(log_B_fourier).detach().cpu())

print("\n--- Real-data Fourier-feature PINN summary ---")
print(f"Position RMSE [km]:          {fourierB_pos_rmse:.4f}")
print(f"Velocity RMSE [m/s]:         {fourierB_vel_rmse:.4f}")
print(f"Final position error [km]:   {fourierB_final_pos_err:.4f}")
print(f"Prior B [m^2/kg]:            {PRIOR_BALLISTIC_COEFF:.4e}")
print(f"Learned B [m^2/kg]:          {fourierB_learned:.4e}  (ratio {fourierB_learned/PRIOR_BALLISTIC_COEFF:.4f})")

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 7))
plt.plot(_truth_pos_km[:, 0], _truth_pos_km[:, 1], label="Dense SGP4 benchmark")
plt.plot(_pred_pos_km[:, 0], _pred_pos_km[:, 1], "--", label="Fourier-PINN reconstruction")
plt.scatter(sparse_obs_df["x_km"], sparse_obs_df["y_km"], s=16, color="k", zorder=5, label="Sparse obs")
plt.xlabel("x [km]"); plt.ylabel("y [km]"); plt.axis("equal")
plt.title("Real ISS orbit: Fourier-PINN reconstruction (x-y)"); plt.legend(); plt.show()

plt.figure()
plt.plot(t_dense / 3600.0, np.linalg.norm(_pred_pos_km - _truth_pos_km, axis=1))
plt.xlabel("Time [hours]"); plt.ylabel("Position error [km]")
plt.title("Real ISS Fourier-PINN reconstruction error over time"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for k in ["total", "data", "physics"]:
    ax[0].semilogy(fourierB_history["epoch"], fourierB_history[k], label=k)
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].set_title("Real-data Fourier-PINN training"); ax[0].legend()
ax[1].plot(fourierB_history["epoch"], fourierB_history["B"], marker="o", label="Learned B")
ax[1].axhline(PRIOR_BALLISTIC_COEFF, ls="--", color="k", label="Prior B")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("B = Cd*A/m [m^2/kg]"); ax[1].set_title("Learned ballistic coefficient"); ax[1].legend()
plt.tight_layout(); plt.show()

_finalB = dict(
    plain_mlp_pos_rmse_km=float(summary_df.loc[summary_df.metric=="position_rmse_km","value"].iloc[0]),
    fourier_pos_rmse_km=fourierB_pos_rmse,
    fourier_vel_rmse_ms=fourierB_vel_rmse,
    fourier_final_pos_err_km=fourierB_final_pos_err,
    prior_B=float(PRIOR_BALLISTIC_COEFF),
    learned_B=fourierB_learned,
    learned_B_over_prior=float(fourierB_learned / PRIOR_BALLISTIC_COEFF),
    classical_baseline_rmse_km=float(baseline_summary.loc[baseline_summary.metric=="baseline_position_rmse_km","value"].iloc[0]) if 'baseline_summary' in dir() else None,
    n_sparse=int(len(sparse_obs_df)), n_dense=int(len(real_dense_df)),
    epochs=FOURIER_EPOCHS_B, k_fourier=24,
)
json.dump(_finalB, open("/tmp/pinn_run/results/nb4_final_metrics.json", "w"), indent=2)
print("\nNB4_FINAL", json.dumps(_finalB, indent=2))


Epoch     1 | total=6.000e+03 | data=1.018e+00 | physics=5.968e+03 | B=5.2381e-03


Epoch   500 | total=1.246e-02 | data=1.220e-04 | physics=1.431e-03 | B=5.2381e-03


Epoch  1000 | total=2.286e-02 | data=1.008e-03 | physics=9.826e-04 | B=5.2381e-03


Epoch  1500 | total=1.438e-04 | data=2.131e-06 | physics=5.384e-05 | B=5.2381e-03


Epoch  2000 | total=2.510e-03 | data=7.710e-05 | physics=7.143e-05 | B=5.2381e-03


...

Epoch  3500 | total=3.302e-04 | data=1.136e-05 | physics=2.315e-05 | B=5.2381e-03


Epoch  4000 | total=1.789e-05 | data=2.051e-07 | physics=8.141e-06 | B=5.2381e-03


Epoch  4500 | total=6.456e-06 | data=2.633e-09 | physics=6.403e-06 | B=5.2381e-03


Epoch  5000 | total=5.224e-06 | data=2.137e-09 | physics=5.180e-06 | B=5.2381e-03

--- Real-data Fourier-feature PINN summary ---
Position RMSE [km]:          5.7627
Velocity RMSE [m/s]:         8.4096
Final position error [km]:   1.5187
Prior B [m^2/kg]:            5.2381e-03
Learned B [m^2/kg]:          5.2381e-03  (ratio 1.0000)



NB4_FINAL {
  "plain_mlp_pos_rmse_km": 7944.3130738167665,
  "fourier_pos_rmse_km": 5.762672979702638,
  "fourier_vel_rmse_ms": 8.409580958558681,
  "fourier_final_pos_err_km": 1.5186716668572182,
  "prior_B": 0.005238095238095238,
  "learned_B": 0.005238096229732037,
  "learned_B_over_prior": 1.0000001893124797,
  "classical_baseline_rmse_km": 16.5714906903511,
  "n_sparse": 25,
  "n_dense": 145,
  "epochs": 5000,
  "k_fourier": 24
}


In [25]:

import json
_m = dict()
for _, r in summary_df.iterrows():
    _m[str(r['metric'])] = float(r['value'])
try:
    for _, r in baseline_summary.iterrows():
        _m[str(r['metric'])] = float(r['value'])
except Exception as e:
    _m['baseline_error'] = str(e)
_m['prior_ballistic_coeff'] = float(PRIOR_BALLISTIC_COEFF)
_m['learned_B'] = float(learned_B)
_m['learned_B_over_prior'] = float(learned_B/PRIOR_BALLISTIC_COEFF)
_m['n_sparse_obs'] = int(len(sparse_obs_df))
_m['n_dense'] = int(len(real_dense_df))
_m['n_collocation'] = int(BASELINE_CONFIG['n_collocation'])
_m['epochs'] = int(BASELINE_CONFIG['epochs'])
_m['real_data_source_note'] = str(real_data_source_note)
_m['target_name'] = str(TARGET_NAME)
_m['dense_min_alt_km'] = float(real_dense_df['altitude_km'].min())
_m['dense_max_alt_km'] = float(real_dense_df['altitude_km'].max())
_m['dense_span_hours'] = float((real_dense_df['t_sec'].max()-real_dense_df['t_sec'].min())/3600.0)
_m['final_total_loss'] = float(history_df['total'].iloc[-1])
json.dump(_m, open('/tmp/pinn_run/results/nb4_metrics.json','w'), indent=2)
print('NB4_METRICS', json.dumps(_m, indent=2))


NB4_METRICS {
  "position_rmse_km": 7944.3130738167665,
  "position_mae_km": 7365.494822394461,
  "final_position_error_km": 2615.781869654803,
  "velocity_rmse_m_per_s": 7367.5039790952205,
  "velocity_mae_m_per_s": 7151.220396760085,
  "prior_ballistic_coeff_m2_per_kg": 0.005238095238095238,
  "learned_ballistic_coeff_m2_per_kg": 0.005238098558038473,
  "learned_B_over_prior_B": 1.0000006338073448,
  "learned_density_scale": 1.0,
  "baseline_position_rmse_km": 16.5714906903511,
  "baseline_final_position_error_km": 28.55386472875776,
  "prior_ballistic_coeff": 0.005238095238095238,
  "learned_B": 0.005238098558038473,
  "learned_B_over_prior": 1.0000006338073448,
  "n_sparse_obs": 25,
  "n_dense": 145,
  "n_collocation": 160,
  "epochs": 500,
  "real_data_source_note": "Downloaded current CelesTrak TLE and propagated with SGP4",
  "target_name": "ISS (ZARYA)",
  "dense_min_alt_km": 412.404077814881,
  "dense_max_alt_km": 423.8919236614465,
  "dense_span_hours": 12.0,
  "final_total_l

In [26]:
print('FIG_FILES', sorted(__import__('os').listdir('figs')))

FIG_FILES ['nb4_fig01.png', 'nb4_fig02.png', 'nb4_fig03.png', 'nb4_fig04.png', 'nb4_fig05.png', 'nb4_fig06.png', 'nb4_fig07.png', 'nb4_fig08.png', 'nb4_fig09.png', 'nb4_fig10.png', 'nb4_fig11.png', 'nb4_fig12.png']
